# Survival Analysis with Progenetix CNV Data

In this notebook, we build a simple survival analysis example using Progenetix data.

The basic workflow:

1. retrieve sample-level metadata,
2. retreive individual-level data with survival information,
3. retrieve sample-level CNV records,
4. convert CNV records into simple gene-level features,
5. and apply two standard survival analysis methods.

We will use:

- **Kaplan–Meier curves** to compare survival between two groups,
- and a **Cox proportional hazards model** to estimate how a CNV-derived feature is associated with hazard.



## Environment Setup

```bash
conda install -c conda-forge scikit-survival
pip install scikit-survival

In [1]:
import requests
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

from tqdm.auto import tqdm

from sksurv.util import Surv
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

## Download sample metadata from Progenetix

We begin with the Progenetix `sampletable` service.

This service returns sample annotations in tabular form and supports query filters.
For example, the Progenetix documentation shows that TCGA samples can be retrieved with:

- `filters=pgx:cohort-TCGAcancers`

and that:

- `limit=0` means no result limit.


In [2]:
# -----------------------------
# Cohort definition
# -----------------------------
TARGET_TCGA_PROJECT = "pgx:TCGA-LUAD"   

# -----------------------------
# Progenetix sampletable endpoint
# -----------------------------
SAMPLETABLE_URL = "https://progenetix.org/services/sampletable/"

In [3]:
def fetch_sampletable(filters, limit=0, timeout=120):
    # Download a Progenetix sample table as a pandas DataFrame.
    params = {
        "filters": filters,
        "limit": limit,
    }

    response = requests.get(SAMPLETABLE_URL, params=params, timeout=timeout)
    response.raise_for_status()

    print("Status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))

    text = response.text
    df = pd.read_csv(io.StringIO(text), sep="\t")
    return df

In [4]:
sample_table_df = fetch_sampletable(filters=TARGET_TCGA_PROJECT, limit=0)

print("Shape:", sample_table_df.shape)
print("Columns:")
print(sample_table_df.columns.tolist())
sample_table_df.head()

Status: 200
Final URL: https://progenetix.org/services/sampletable/?filters=pgx%3ATCGA-LUAD&limit=0
Content-Type: text/tsv
Shape: (1110, 42)
Columns:
['biosample_id', 'individual_id', 'biosample_name', 'notes', 'histological_diagnosis_id', 'histological_diagnosis_label', 'pathological_stage_id', 'pathological_stage_label', 'biosample_status_id', 'biosample_status_label', 'sample_origin_type_id', 'sample_origin_type_label', 'sampled_tissue_id', 'sampled_tissue_label', 'tnm', 'tumor_grade_id', 'tumor_grade_label', 'age_iso', 'age_days', 'icdo_morphology_id', 'icdo_morphology_label', 'icdo_topography_id', 'icdo_topography_label', 'pubmed_id', 'pubmed_label', 'cellosaurus_id', 'cellosaurus_label', 'cbioportal_id', 'cbioportal_label', 'tcgaproject_id', 'tcgaproject_label', 'cohorts', 'geoprov_id', 'geoprov_city', 'geoprov_country', 'geoprov_iso_alpha3', 'geoprov_long_lat', 'followup_days', 'sex_id', 'sex_label', 'group_id', 'group_label']


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,biosample_status_id,biosample_status_label,...,geoprov_id,geoprov_city,geoprov_country,geoprov_iso_alpha3,geoprov_long_lat,followup_days,sex_id,sex_label,group_id,group_label
0,pgxbs-kftvhm7v,pgxind-kftx3khu,180eb213-0ddc-4f10-ab0e-b3af92cde3c3,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27977,Stage IIIA,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
1,pgxbs-kftvi6te,pgxind-kftx3gzq,d635ce37-1e71-44f5-b415-76ab7b5b7bfc,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27976,Stage IB,EFO:0009654,reference sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
2,pgxbs-kftvi0cd,pgxind-kftx3jbn,13b0d336-205a-42ac-8a50-db0eb065013d,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
3,pgxbs-kftvhy7t,pgxind-kftx3s2n,558e490b-6615-43e2-b954-7c1633b48822,Solid Tissue Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,EFO:0009654,reference sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN
4,pgxbs-kftvhmru,pgxind-kftx3kx2,d2198941-e96f-40bd-9fbe-82886217d5db,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,EFO:0009656,neoplastic sample,...,cambridge::unitedstates::-71.10561::42.3751,Cambridge,United States,USA,-71.10561::42.3751,NaN,NaN,NaN,NaN,NaN


In [5]:
# Columns we want to keep from the sample table
BIOSAMPLE_COLUMNS = [
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",
]

BIOSAMPLE_COLUMNS = [c for c in BIOSAMPLE_COLUMNS if c in sample_table_df.columns]

biosample_df = sample_table_df[BIOSAMPLE_COLUMNS].copy()

print("Biosample-level metadata shape:", biosample_df.shape)
biosample_df.head()

Biosample-level metadata shape: (1110, 17)


,biosample_id,individual_id,biosample_name,notes,histological_diagnosis_id,histological_diagnosis_label,pathological_stage_id,pathological_stage_label,sample_origin_type_id,sample_origin_type_label,sampled_tissue_id,sampled_tissue_label,tcgaproject_id,tcgaproject_label,icdo_morphology_id,icdo_topography_id,icdo_topography_label
0,pgxbs-kftvhm7v,pgxind-kftx3khu,180eb213-0ddc-4f10-ab0e-b3af92cde3c3,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27977,Stage IIIA,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"
1,pgxbs-kftvi6te,pgxind-kftx3gzq,d635ce37-1e71-44f5-b415-76ab7b5b7bfc,Blood Derived Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,UBERON:0000178,blood,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-00000,pgx:icdot-C42.0,Blood
2,pgxbs-kftvi0cd,pgxind-kftx3jbn,13b0d336-205a-42ac-8a50-db0eb065013d,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"
3,pgxbs-kftvhy7t,pgxind-kftx3s2n,558e490b-6615-43e2-b954-7c1633b48822,Solid Tissue Normal,NCIT:C132256,Unspecified Tissue,NCIT:C27975,Stage IA,OBI:0001479,specimen from organism,UBERON:0001062,anatomical entity,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-00000,pgx:icdot-C80.9,Unknown primary site
4,pgxbs-kftvhmru,pgxind-kftx3kx2,d2198941-e96f-40bd-9fbe-82886217d5db,Primary Tumor,NCIT:C3512,Lung Adenocarcinoma,NCIT:C27976,Stage IB,OBI:0001479,specimen from organism,UBERON:0008948,upper lobe of lung,pgx:TCGA-LUAD,Lung Adenocarcinoma,pgx:icdom-81403,pgx:icdot-C34.1,"Upper lobe, lung"


## Build survival metadata by combining biosample- and individual-level information

The Progenetix sample table is useful for cohort selection and biosample-level metadata, but survival-related information is not always available there.

In our case, each biosample is linked to an `individual_id`.  
We therefore use a two-step strategy:

1. retrieve **biosample-level metadata** from the Progenetix sample table,
2. retrieve **individual-level follow-up metadata** from the Progenetix individual endpoint,
3. and merge the two through `individual_id`.

This is important because survival analysis requires at least:

- a follow-up time,
- and an event indicator.

These fields are available in the individual-level record.

In the Progenetix data model, one individual may have more than one biosample.  
Therefore, after merging biosample- and individual-level metadata, we also need to decide how to handle the one-to-many relationship before building a survival model.

In [6]:
individual_ids = (
    biosample_df["individual_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Number of unique individuals:", len(individual_ids))
print(individual_ids[:10])

Number of unique individuals: 518
['pgxind-kftx3khu', 'pgxind-kftx3gzq', 'pgxind-kftx3jbn', 'pgxind-kftx3s2n', 'pgxind-kftx3kx2', 'pgxind-kftx3ndg', 'pgxind-kftx3qln', 'pgxind-kftx3lgl', 'pgxind-kftx3grm', 'pgxind-kftx3i69']


In [7]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def fetch_individual_record(individual_id, session=SESSION, timeout=120):
    # Fetch one Progenetix individual record by individual ID.
    url = f"https://progenetix.org/beacon/individuals/{individual_id}/"
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

In [8]:
def parse_individual_record(data):
    # Parse one Progenetix individual JSON response into a flat dictionary.
    response = data.get("response", {})
    result_sets = response.get("resultSets", [])
    if not result_sets:
        return None

    results = result_sets[0].get("results", [])
    if not results:
        return None

    rec = results[0]

    diseases = rec.get("diseases", [])
    disease_0 = diseases[0] if len(diseases) > 0 else {}

    out = {
        # identifiers
        "individual_id": rec.get("id"),
        "n_diseases": len(diseases),

        # sex
        "sex_id_individual": rec.get("sex", {}).get("id"),
        "sex_label_individual": rec.get("sex", {}).get("label"),

        # top-level survival-related status
        "vital_status": rec.get("vitalStatus", {}).get("status"),

        # top-level info block
        "info_age_at_diagnosis_days": rec.get("info", {}).get("ageAtDiagnosis"),
        "info_days_to_death": rec.get("info", {}).get("daysToDeath"),
        "info_death": rec.get("info", {}).get("death"),
        "info_ethnicity": rec.get("info", {}).get("ethnicity"),
        "info_race": rec.get("info", {}).get("race"),
        "info_year_of_birth": rec.get("info", {}).get("yearOfBirth"),

        # references
        "tcga_case_id": rec.get("references", {}).get("tcgacase", {}).get("id"),
        "tcga_submitter_id": rec.get("references", {}).get("tcgasubmitter", {}).get("id"),

        # disease block 
        "disease_id": disease_0.get("diseaseCode", {}).get("id"),
        "disease_label": disease_0.get("diseaseCode", {}).get("label"),
        "followup_days": disease_0.get("followupDays"),
        "followup_time_iso": disease_0.get("followupTime"),
        "followup_state_id": disease_0.get("followupState", {}).get("id"),
        "followup_state_label": disease_0.get("followupState", {}).get("label"),
        "onset_age_iso": disease_0.get("onset", {}).get("age"),
        "onset_age_days": disease_0.get("onset", {}).get("ageDays"),
        "stage_id_individual": disease_0.get("stage", {}).get("id"),
        "stage_label_individual": disease_0.get("stage", {}).get("label"),
    }

    return out

In [ ]:
individual_rows = []
failed_individuals = []

for individual_id in tqdm(individual_ids, desc="Fetching individual records"):
    try:
        data = fetch_individual_record(individual_id)
        row = parse_individual_record(data)
        if row is not None:
            individual_rows.append(row)
        else:
            failed_individuals.append((individual_id, "No result"))
    except Exception as e:
        failed_individuals.append((individual_id, str(e)))

individual_df = pd.DataFrame(individual_rows)

print("Fetched individual rows:", len(individual_df))
print("Failed individual queries:", len(failed_individuals))
individual_df.head()

Fetching individual records:   0%|          | 0/518 [00:00<?, ?it/s]

### Merge Biosample and Individual with individual_id

In [ ]:
merged_meta_df = biosample_df.merge(
    individual_df,
    on="individual_id",
    how="left"
)

print("Merged metadata shape:", merged_meta_df.shape)
merged_meta_df.head()

### Build Survival Outcome
Keep one biosample for each individual

In [ ]:
survival_df = merged_meta_df.copy()

# Survival time
survival_df["time"] = pd.to_numeric(survival_df["followup_days"], errors="coerce")

def map_event(row):
    """
    Convert individual-level follow-up / vital status into a binary event label.
    0 = alive / censored
    1 = dead / event occurred
    """
    followup_state = str(row.get("followup_state_label", "")).strip().lower()
    vital_status = str(row.get("vital_status", "")).strip().upper()

    if "alive" in followup_state:
        return 0
    if "deceased" in followup_state or "dead" in followup_state:
        return 1

    if vital_status == "ALIVE":
        return 0
    if vital_status == "DECEASED":
        return 1

    return np.nan

survival_df["event"] = survival_df.apply(map_event, axis=1)

# Keep only rows with usable survival data
survival_df = survival_df.dropna(subset=["biosample_id", "individual_id", "time", "event"]).copy()
survival_df["event"] = survival_df["event"].astype(int)

print("Rows with usable survival data:", len(survival_df))
survival_df[[
    "biosample_id",
    "individual_id",
    "time",
    "event",
    "followup_state_label",
    "vital_status",
    "sex_label_individual"
]].head()

In [ ]:
survival_unique_df = (
    survival_df
    .groupby("individual_id")
    .sample(n=1, random_state=42)   
    .copy()
)

print("Rows after keeping one biosample per individual:", len(survival_unique_df))
survival_unique_df[[
    "biosample_id",
    "individual_id",
    "sample_origin_type_label",
    "time",
    "event",
    "sex_label_individual"
]].head()

In [ ]:
FINAL_META_COLUMNS = [
    # biosample-level
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",

    # individual-level
    "sex_id_individual",
    "sex_label_individual",
    "vital_status",
    "info_age_at_diagnosis_days",
    "info_days_to_death",
    "info_death",
    "info_ethnicity",
    "info_race",
    "info_year_of_birth",
    "tcga_case_id",
    "tcga_submitter_id",
    "disease_id",
    "disease_label",
    "followup_days",
    "followup_time_iso",
    "followup_state_id",
    "followup_state_label",
    "onset_age_iso",
    "onset_age_days",
    "stage_id_individual",
    "stage_label_individual",

    # derived survival variables
    "time",
    "event",
]

FINAL_META_COLUMNS = [c for c in FINAL_META_COLUMNS if c in survival_unique_df.columns]

survival_unique_df = survival_unique_df[FINAL_META_COLUMNS].copy()

print("Final survival metadata shape:", survival_unique_df.shape)
survival_unique_df.head()

## Retrieve full CNV records and derive gene-level features locally

We use the record-level endpoint:

- retrieve all CNV records for each retained biosample,
- identify which CNV intervals overlap the target gene,
- and summarize those overlaps into simple binary features.

For the target gene, we define:

- `EGFR_hit`: at least one CNV overlaps the EGFR locus
- `EGFR_gain_hit`: at least one overlapping CNV is a gain
- `EGFR_loss_hit`: at least one overlapping CNV is a loss



In [ ]:
TARGET_GENE = "EGFR"

GENE_COORDS = {
    "EGFR": {
        "chromosome": "7",
        "start": 55019017,
        "end": 55211628,
    }
}

gene_info = GENE_COORDS[TARGET_GENE]
print(TARGET_GENE, gene_info)

## Define helper functions

We now define helper functions to:

1. retrieve full CNV records for one biosample,
2. extract the record list from the Beacon response,
3. test whether a CNV interval overlaps the target gene,
4. classify the CNV state as gain or loss,
5. and summarize all records for one biosample into three binary gene-level features.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def fetch_biosample_variants_record(biosample_id, session=SESSION, timeout=120):
    """
    Fetch all CNV variant records for one biosample from the
    biosample-scoped Progenetix g_variants endpoint.
    """
    url = f"https://progenetix.org/beacon/biosamples/{biosample_id}/g_variants/"
    params = {
        "requestedGranularity": "record"
    }
    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()

In [ ]:
def extract_variant_results(data):
    # Extract the list of variant records from a Beacon JSON response.
    response = data.get("response", {})
    result_sets = response.get("resultSets", [])

    if not result_sets:
        return []

    return result_sets[0].get("results", [])

In [ ]:
def intervals_overlap(start1, end1, start2, end2):
    # Return True if two genomic intervals overlap.
    return (start1 <= end2) and (start2 <= end1)


def classify_copy_change(label):
    """
    Reduce CNV state labels into broader categories.

    Output categories:
    - gain
    - loss
    - other
    """
    if not isinstance(label, str):
        return "other"

    x = label.strip().lower()

    if "gain" in x or "high-level gain" in x or "low-level gain" in x:
        return "gain"
    if "loss" in x or "high-level loss" in x or "low-level loss" in x:
        return "loss"

    return "other"

## Summarize one biosample into EGFR-based binary features

For each biosample, we scan all returned CNV records and check:

- whether the record is on the same chromosome as EGFR,
- whether the CNV interval overlaps the EGFR locus,
- and whether the overlapping CNV is classified as gain or loss.

From this, we build three simple Boolean features:

- `EGFR_hit`
- `EGFR_gain_hit`
- `EGFR_loss_hit`

In [ ]:
def summarize_gene_features_from_records(records, gene_name, gene_info):
    """
    Summarize all CNV records for one biosample into three binary features:
    - GENE_hit
    - GENE_gain_hit
    - GENE_loss_hit
    """
    gene_chr = str(gene_info["chromosome"])
    gene_start = int(gene_info["start"])
    gene_end = int(gene_info["end"])

    features = {
        f"{gene_name}_hit": 0,
        f"{gene_name}_gain_hit": 0,
        f"{gene_name}_loss_hit": 0,
    }

    for item in records:
        variation = item.get("variation", {})
        location = variation.get("location", {})

        chrom = str(location.get("chromosome"))
        start = location.get("start")
        end = location.get("end")

        if chrom != gene_chr:
            continue
        if start is None or end is None:
            continue

        if not intervals_overlap(start, end, gene_start, gene_end):
            continue

        # At least one overlapping CNV exists
        features[f"{gene_name}_hit"] = 1

        # Prefer copyChange, fallback to variantState.label
        state_label = variation.get("copyChange")
        if state_label is None:
            state_label = variation.get("variantState", {}).get("label")

        state = classify_copy_change(state_label)

        if state == "gain":
            features[f"{gene_name}_gain_hit"] = 1
        elif state == "loss":
            features[f"{gene_name}_loss_hit"] = 1

    return features

In [ ]:
def inspect_gene_overlaps_for_one_biosample(biosample_id, gene_name, gene_info):
    # Debug helper: print overlapping CNV records for one biosample.

    data = fetch_biosample_variants_record(biosample_id)
    records = extract_variant_results(data)

    gene_chr = str(gene_info["chromosome"])
    gene_start = int(gene_info["start"])
    gene_end = int(gene_info["end"])

    overlaps = []

    for item in records:
        variation = item.get("variation", {})
        location = variation.get("location", {})

        chrom = str(location.get("chromosome"))
        start = location.get("start")
        end = location.get("end")

        if chrom != gene_chr:
            continue
        if start is None or end is None:
            continue
        if not intervals_overlap(start, end, gene_start, gene_end):
            continue

        overlaps.append({
            "chromosome": chrom,
            "start": start,
            "end": end,
            "copyChange": variation.get("copyChange"),
            "variantStateLabel": variation.get("variantState", {}).get("label"),
        })

    return pd.DataFrame(overlaps)

In [ ]:
test_biosample_id = survival_unique_df["biosample_id"].iloc[0]
inspect_gene_overlaps_for_one_biosample(test_biosample_id, TARGET_GENE, gene_info)

## Build EGFR features for all retained biosamples

We now apply the same logic to all retained biosamples.

For each biosample, we:
- retrieve the full CNV record set once,
- derive EGFR overlap features locally,
- and store the results in a compact biosample-level feature table.